# 10 · Comparison
**Owner:** Nimnadi · **Branch:** `compare/10-comparison`

Collect every model's metrics (merged from the model branches), rank by test MAE, plot predicted vs true, and show feature importance for the best model.


In [ ]:
# --- Colab setup (run first) ---
import os
REPO = '/content/ev-battery-rul-prediction'   # absolute path -> safe to re-run
if not os.path.exists(REPO):
    !git clone https://github.com/Chathumi-Senanayake/ev-battery-rul-prediction.git {REPO}
%cd {REPO}
!git pull -q
!pip install -q -r requirements.txt
import sys
if REPO not in sys.path: sys.path.insert(0, REPO)

# Try Google Drive (data persists across sessions). If auth fails, fall back
# to local Colab storage so you are never blocked.
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = '/content/drive/MyDrive/ev_battery_data'
except Exception as e:
    print('Drive mount failed -> using local storage:', e)
    DATA_DIR = '/content/ev_battery_data'
os.makedirs(DATA_DIR, exist_ok=True)
print('DATA_DIR =', DATA_DIR)


In [ ]:
import glob, json, pandas as pd
rows = [json.load(open(p)) for p in sorted(glob.glob(os.path.join(DATA_DIR,'metrics_*.json')))]
res = pd.DataFrame(rows).sort_values('test_MAE')
res


In [ ]:
import matplotlib.pyplot as plt
fig,ax=plt.subplots(figsize=(7,4))
ax.barh(res['model'], res['test_MAE']); ax.invert_yaxis()
ax.set_xlabel('test MAE (cycles)'); ax.set_title('model comparison'); plt.tight_layout(); plt.show()


In [ ]:
# predicted vs true for the best model
best = res.iloc[0]['model']; num = res.iloc[0].name
# load its pred csv (pred_XX.csv) and scatter RUL vs pred
import glob
pred_files = {json.load(open(p))['model']: p.replace('metrics_','pred_').replace('.json','.csv')
              for p in glob.glob(os.path.join(DATA_DIR,'metrics_*.json'))}
pf = pred_files.get(best)
if pf and os.path.exists(pf):
    d = pd.read_csv(pf)
    plt.scatter(d['RUL'], d['pred'], s=4, alpha=.3)
    lim=[d['RUL'].min(), d['RUL'].max()]; plt.plot(lim,lim,'r--')
    plt.xlabel('true RUL'); plt.ylabel('predicted'); plt.title(f'{best}: pred vs true'); plt.show()
